In [27]:
%pip install pandas numpy tqdm regex rapidfuzz

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
#==========================================================
# IMPORT LIBRARIES
#==========================================================

import os
import re
import csv
import json
import logging
import warnings

import pandas as pd
import numpy as np

from tqdm import tqdm
from glob import glob
from rapidfuzz import fuzz

warnings.filterwarnings("ignore")

In [30]:
#==========================================================
# INITIALIZE DIRECTORY
#==========================================================

ROOT_DIR = "/content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan"

PATH_RAW = os.path.join(
    ROOT_DIR,
    "data",
    "raw_text"
)

PATH_PROCESSED = os.path.join(
    ROOT_DIR,
    "data",
    "processed"
)

PATH_CSV = os.path.join(
    PATH_PROCESSED,
    "cases.csv"
)

PATH_JSON = os.path.join(
    PATH_PROCESSED,
    "cases.json"
)

LOG_DIR = os.path.join(
    ROOT_DIR,
    "logs"
)

LOG_PATH = os.path.join(
    LOG_DIR,
    "metadata_extraction.log"
)

os.makedirs(PATH_PROCESSED, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("ROOT DIR :", ROOT_DIR)
print("RAW TEXT :", PATH_RAW)
print("CSV PATH :", PATH_CSV)

ROOT DIR : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan
RAW TEXT : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/data/raw_text
CSV PATH : /content/drive/MyDrive/2023-453_PenalaranKomputerB/SubCPMK3/perikanan/data/processed/cases.csv


In [31]:
#==========================================================
# INITIALIZE LOGGING
#==========================================================

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(

    filename=LOG_PATH,

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    filemode="w"
)

logging.info("===== NOTEBOOK 02 STARTED =====")

In [32]:
#==========================================================
# NATURAL SORT
#==========================================================

def natural_sort_key(path):

    filename = os.path.basename(path)

    numbers = re.findall(r'\d+', filename)

    return int(numbers[0]) if numbers else 0

In [33]:
#==========================================================
# LOAD TXT FILES
#==========================================================

text_files = glob(
    os.path.join(PATH_RAW, "*.txt")
)

text_files = sorted(
    text_files,
    key=natural_sort_key
)

print(f"TOTAL TXT FILES : {len(text_files)}")

logging.info(
    f"TOTAL TXT FILES : {len(text_files)}"
)

TOTAL TXT FILES : 30


In [34]:
#==========================================================
# TEXT CLEANING
#==========================================================
def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    patterns = [

        r'Direktori Putusan.*?RI',

        r'Mahkamah Agung Republik Indonesia',

        r'putusan\.mahkamahagung\.go\.id',

        r'Disclaimer',

        r'Kepaniteraan.*?RI',

        r'Email\s*:.*',

        r'Telp\s*:.*',

        r'Hal\.\s*\d+',

        r'waktu kewaktu.*?hubungi melalui',

        r'dalam hal anda menemukan inakurasi informasi.*?hubungi melalui',

        r'halaman \d+ dari \d+ hal',

    ]

    for pattern in patterns:

        text = re.sub(
            pattern,
            ' ',
            text,
            flags=re.IGNORECASE
        )

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [35]:
#==========================================================
# EXTRACT NOMOR PUTUSAN
#==========================================================

def clean_nomor_putusan(text):

    text = text.upper()

    text = text.replace(
        "PID. SUS",
        "PID.SUS"
    )

    text = re.sub(
        r'[^A-Z0-9./\-]',
        '',
        text
    )

    return text.strip()


def extract_nomor_putusan(text):

    patterns = [

        r'Nomor\s*:?\s*([0-9]+[\w./\-]+)',

        r'PUTUSAN\s+Nomor\s*:?\s*([0-9]+[\w./\-]+)',

        r'([0-9]+K/PID\.?SUS/[0-9]{4})',

        r'([0-9]+/Pid\.[A-Za-z./\-]+/[0-9]{4}/PN\.[A-Za-z]+)',

        r'([0-9]{1,5}/[A-Za-z0-9.\-]+/[0-9]{4})',
    ]

    candidates = []

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text[:10000],
            re.IGNORECASE
        )

        for match in matches:

            nomor = clean_nomor_putusan(match)

            if "/" in nomor:
                candidates.append(nomor)

    candidates = list(dict.fromkeys(candidates))

    if candidates:

        candidates = sorted(
            candidates,
            key=len,
            reverse=True
        )

        return candidates[0]

    return "Tidak ditemukan"

In [36]:
#==========================================================
# EXTRACT TAHUN
#==========================================================

def extract_tahun(text):

    match = re.search(
        r'/(\d{4})/',
        text
    )

    if match:
        return match.group(1)

    return "Tidak ditemukan"

In [37]:
#==========================================================
# EXTRACT PENGADILAN
#==========================================================

def extract_pengadilan(text):

    patterns = [

        r'Pengadilan Negeri\s+[A-Za-z]+',

        r'Pengadilan Tinggi\s+[A-Za-z]+',
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if match:

            hasil = match.group(0)

            hasil = re.sub(
                r'(karena|yang|dan).*',
                '',
                hasil,
                flags=re.IGNORECASE
            )

            return hasil.strip()

    if "Mahkamah Agung" in text:
        return "Mahkamah Agung"

    return "Tidak ditemukan"

In [38]:
#==========================================================
# EXTRACT TERDAKWA
#==========================================================

def clean_name(name):

    name = name.upper()

    stopwords = [

        "TEMPAT LAHIR",

        "UMUR",

        "TANGGAL LAHIR",

        "JENIS KELAMIN",

        "KEBANGSAAN",

        "TEMPAT TINGGAL",

        "AGAMA",

        "PEKERJAAN",
    ]

    for stop in stopwords:

        if stop in name:
            name = name.split(stop)[0]

    name = re.sub(
        r'[^A-Z\s.]',
        ' ',
        name
    )

    name = re.sub(
        r'\s+',
        ' ',
        name
    )

    return name.strip()


def extract_terdakwa(text):

    hasil = []

    patterns = [

        r'Nama lengkap\s*:\s*(.*?);',

        r'Nama\s*:\s*(.*?);',

        r'Terdakwa\s*:\s*(.*?);',
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text[:15000],
            re.IGNORECASE | re.DOTALL
        )

        for match in matches:

            nama = clean_name(match)

            if len(nama) > 3:
                hasil.append(nama)

    hasil = list(dict.fromkeys(hasil))

    if hasil:
        return " | ".join(hasil[:10])

    return "Tidak ditemukan"

In [39]:
#==========================================================
# EXTRACT PASAL
#==========================================================

def extract_pasal(text):

    patterns = [

        r'pasal\s+\d+\s+ayat\s+\(\d+\)',

        r'pasal\s+\d+',
    ]

    hasil = []

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            re.IGNORECASE
        )

        hasil.extend(matches)

    hasil = list(set(hasil))

    if hasil:
        return "; ".join(hasil[:10])

    return "Tidak ditemukan"

In [40]:
#==========================================================
# EXTRACT DAKWAAN
#==========================================================

def extract_dakwaan(text):

    keywords = [

        "didakwa",

        "dakwaan",

        "kesatu",
    ]

    lower = text.lower()

    for keyword in keywords:

        idx = lower.find(keyword)

        if idx != -1:

            chunk = text[idx:idx+5000]

            chunk = normalize_text(chunk)

            if len(chunk) > 100:
                return chunk

    return "Tidak ditemukan"

In [41]:
# =========================================================
# EXTRACT RINGKASAN FAKTA
# =========================================================

def extract_ringkasan_fakta(text):

    text_lower = text.lower()

    patterns = [

        r'bahwa terdakwa.*?(?=menimbang)',

        r'berawal dari.*?(?=menimbang)',

        r'pada hari.*?(?=menimbang)',

        r'telah melakukan.*?(?=menimbang)',

        r'jaksa penuntut umum.*?(?=menimbang)',

        r'didakwa.*?(?=menimbang)',

        r'uraian kejadian.*?(?=menimbang)',
    ]

    for pattern in patterns:

        match = re.search(

            pattern,

            text_lower,

            re.DOTALL | re.IGNORECASE
        )

        if match:

            fakta = match.group(0)

            fakta = re.sub(r'\s+', ' ', fakta)

            fakta = fakta.strip()

            if len(fakta) > 100:

                logging.info(
                    f"Ringkasan fakta ditemukan"
                )

                return fakta[:1500]

    logging.warning(
        "Ringkasan fakta tidak ditemukan"
    )

    return ""

In [42]:
#==========================================================
# EXTRACT AMAR PUTUSAN
#==========================================================

def extract_amar_putusan(text):

    keywords = [

        "mengadili",

        "memutuskan",

        "menyatakan terdakwa",
    ]

    lower = text.lower()

    for keyword in keywords:

        idx = lower.find(keyword)

        if idx != -1:

            chunk = text[idx:idx+4000]

            chunk = normalize_text(chunk)

            if len(chunk) > 100:
                return chunk

    return "Tidak ditemukan"

In [43]:
#==========================================================
# EXTRACT JENIS PUTUSAN
#==========================================================

def extract_jenis_putusan(text):

    text = text.lower()

    if any(k in text for k in [
        "bom",
        "bahan peledak",
        "peledak"
    ]):
        return "Bom Ikan"

    if any(k in text for k in [
        "trawl",
        "pukat harimau"
    ]):
        return "Trawl"

    if "sipi" in text:
        return "Tanpa SIPI"

    if "siup" in text:
        return "Tanpa SIUP"

    if "spb" in text:
        return "Tanpa SPB"

    if any(k in text for k in [
        "vietnam",
        "thailand",
        "asing"
    ]):
        return "Kapal Asing"

    if "illegal fishing" in text:
        return "Illegal Fishing"

    return "Perikanan Umum"

In [44]:
#==========================================================
# BUILD DATASET
#==========================================================

dataset = []

for file_path in tqdm(text_files):

    filename = os.path.basename(file_path)

    logging.info(
        f"PROCESSING : {filename}"
    )

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

    except:

        with open(
            file_path,
            "r",
            encoding="latin-1"
        ) as f:

            text = f.read()

    text = normalize_text(text)

    case_data = {

        "case_id":
        filename.replace(".txt", ""),

        "nomor_putusan":
        extract_nomor_putusan(text),

        "tahun":
        extract_tahun(text),

        "pengadilan":
        extract_pengadilan(text),

        "terdakwa":
        extract_terdakwa(text),

        "pasal":
        extract_pasal(text),

        "jenis_putusan":
        extract_jenis_putusan(text),

        "dakwaan":
        extract_dakwaan(text),

        "ringkasan_fakta":
        extract_ringkasan_fakta(text),

        "amar_putusan":
        extract_amar_putusan(text),

        "full_text":
        text
    }

    dataset.append(case_data)

    logging.info(
        f"SUCCESS : {filename}"
    )

print("DATASET CREATED")

100%|██████████| 30/30 [00:01<00:00, 28.18it/s]

DATASET CREATED


In [45]:
#==========================================================
# CREATE DATAFRAME
#==========================================================

df = pd.DataFrame(dataset)
df["retrieval_topic"] = df["jenis_putusan"]

print(df.shape)

df.head()

(30, 12)


,case_id,nomor_putusan,tahun,pengadilan,terdakwa,pasal,jenis_putusan,dakwaan,ringkasan_fakta,amar_putusan,full_text,retrieval_topic
0,case_001,1105/2016/S.270/TAH.SUS.IK/PP/2016/MATANGGAL29,2016,Pengadilan Negeri sejak,SOPIAN BINRABBIL | SARIF BINCU | SAMSULBINKOHE...,Pasal 8 ayat (1); Pasal 9; Pasal 46; Pasal 39;...,Bom Ikan,didakwa:KESATU :BahwaTerdakwaISOPIAN binRABBIL...,pada hari senin tanggal 02 nopember 2015sekira...,"mengadili,“Mereka yang melakukan, yang menyuru...",: Halaman 1 dari13hal. Put. Nomor :497K/Pid.Su...,Bom Ikan
1,case_002,7/AKTA-KASASI.PID.SUS-PRK/2017/PN.MDN,2017,Pengadilan Negeri Me,RIZAL | NAMA RIZAL,Pasal 9 ayat (1); Pasal 85; Pasal 98; Pasal 9,Trawl,didakwa dengan dakwaan sebagai berikut:Dakwaan...,bahwa terdakwa melanggar pasal 98 undang-undan...,MENGADILI SENDIRI:1.Menyatakan Terdakwa Rizal ...,: Halaman 1 PUTUSANNomor 102 K/Pid.Sus/2018DEM...,Trawl
2,case_003,3370/2016/S.853.TAH.SUS.IK/PP/2016/MA.TANGGAL,2016,Pengadilan Negeri Palembang,GINDA PURNAMABIN TEGIN | NAMA GINDA PURNAMABIN...,Pasal 30; Pasal 162; Pasal 253 Ayat (1); Pasal...,Trawl,didakwa dengan dakwaan sebagai berikut :Bahwa ...,"pada hari kamis,tanggal 4 februari 2016 sekira...",mengadili Pengadilan NegeriPalembang dikarenak...,: Halaman 1 dari18hal. Put. No.1442K/PID.SUS/2...,Trawl
3,case_004,88/PID.SUS/2015/PN.WNO.,2015,Pengadilan Negeri Wonosari,N A M A LENGKAP SUGIANTORO BIN SAINO ALIAS GIANTO,pasal 42 ayat (3); pasal 27 ayat (1); pasal 98...,Tanpa SIPI,didakwa sebagai berikut : Kesatu: Bahwa terdak...,bahwa terdakwa sugiyantoro als. giyanto bin sa...,mengadili Perkara Pidana dalam tingkat banding...,: Halaman 1 Nomor 84/PID.SUS/2015/PT YYK. PUTU...,Tanpa SIPI
4,case_005,24/AKTA.PID.SUS-PRK/2018/PNRANJUNCTONOMOR,2018,Pengadilan Negeri Ranai,TRAN VAN TRUONG | NAMA TRAN VAN TRUONG,Pasal 9; Pasal 102; Pasal 55; Pasal 92; Pasal ...,Trawl,didakwa dengan dakwaan sebagai berikut:-Kesatu...,bahwa terdakwa mengakuibelumpernah dihukum sed...,mengadili sendiri perkara inidengan amar putus...,: Halaman 1 dari9hal. Putusan Nomor1335K/Pid.S...,Trawl


In [46]:
#==========================================================
# SAVE CSV
#==========================================================

df.to_csv(

    PATH_CSV,

    index=False,

    encoding="utf-8-sig"
)

print("CSV SAVED")

logging.info(
    f"CSV SAVED : {PATH_CSV}"
)

CSV SAVED


In [47]:
#==========================================================
# SAVE JSON
#==========================================================

with open(
    PATH_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dataset,
        f,
        ensure_ascii=False,
        indent=4
    )

print("JSON SAVED")

logging.info(
    f"JSON SAVED : {PATH_JSON}"
)

JSON SAVED


In [48]:
#==========================================================
# DATA QUALITY CHECK
#==========================================================

print("="*60)
print("DATASET SHAPE")
print("="*60)

print(df.shape)

print("\n")

print("="*60)
print("MISSING VALUES")
print("="*60)

print(df.isnull().sum())

print("\n")

print("="*60)
print("JENIS PUTUSAN")
print("="*60)

print(
    df["jenis_putusan"].value_counts()
)

DATASET SHAPE
(30, 12)


MISSING VALUES
case_id            0
nomor_putusan      0
tahun              0
pengadilan         0
terdakwa           0
pasal              0
jenis_putusan      0
dakwaan            0
ringkasan_fakta    0
amar_putusan       0
full_text          0
retrieval_topic    0
dtype: int64


JENIS PUTUSAN
jenis_putusan
Bom Ikan          8
Trawl             8
Tanpa SIPI        8
Perikanan Umum    3
Kapal Asing       2
Tanpa SIUP        1
Name: count, dtype: int64


In [49]:
#==========================================================
# SAMPLE RESULT
#==========================================================

sample = df.iloc[0]

for col in df.columns:

    print("="*80)

    print(col.upper())

    print("="*80)

    print(
        str(sample[col])[:2000]
    )

    print("\n")

CASE_ID
case_001


NOMOR_PUTUSAN
1105/2016/S.270/TAH.SUS.IK/PP/2016/MATANGGAL29


TAHUN
2016


PENGADILAN
Pengadilan Negeri sejak


TERDAKWA
SOPIAN BINRABBIL | SARIF BINCU | SAMSULBINKOHER | I.NAMA LENGKAP SOPIAN BINRABBIL


PASAL
Pasal 8 ayat (1); Pasal 9; Pasal 46; Pasal 39; Pasal 55; Pasal 84 ayat (1); Pasal 9 ayat (1); Pasal 55 ayat (1); Pasal 85; Pasal 76


JENIS_PUTUSAN
Bom Ikan


DAKWAAN
didakwa:KESATU :BahwaTerdakwaISOPIAN binRABBIL,TerdakwaIISARIF bin CUdanTerdakwaIIISAMSUL bin KOHER, pada hari Senin tanggal 02 Nopember 2015sekira pukul 12.15 wita atau setidak-tidaknya pada suatu waktu tertentu dalambulan Nopember tahun 2015, bertempat di Perairan laut Kandolo BontangKalimantan Timur atau setidak-tidaknya pada suatu tempat lain yang masihtermasuk dalam daerah hukum Pengadilan Negeri Bontang yang berwenangmemeriksa dan mengadili,“Mereka yang melakukan, yang menyuruhmelakukan, dan yang turut serta melakukan perbuatan yang dengan sengaja diwilayah pengelolaan perikanan Republik I

In [50]:
#==========================================================
# FINISH LOGGING
#==========================================================

logging.info(
    "===== NOTEBOOK 02 FINISHED ====="
)

logging.shutdown()

print("NOTEBOOK 02 COMPLETED")

NOTEBOOK 02 COMPLETED


In [51]:
df[
[
    "case_id",
    "jenis_putusan",
    "retrieval_topic"
]
].head(30)

,case_id,jenis_putusan,retrieval_topic
0,case_001,Bom Ikan,Bom Ikan
1,case_002,Trawl,Trawl
2,case_003,Trawl,Trawl
3,case_004,Tanpa SIPI,Tanpa SIPI
4,case_005,Trawl,Trawl
5,case_006,Bom Ikan,Bom Ikan
6,case_007,Tanpa SIUP,Tanpa SIUP
7,case_008,Perikanan Umum,Perikanan Umum
8,case_009,Tanpa SIPI,Tanpa SIPI
9,case_010,Bom Ikan,Bom Ikan


In [52]:
df["retrieval_topic"].value_counts()

,count
retrieval_topic,
Bom Ikan,8
Trawl,8
Tanpa SIPI,8
Perikanan Umum,3
Kapal Asing,2
Tanpa SIUP,1
